# AI Workforce Intelligence Platform

# Data Preprocessing

## Objective

The objective of this notebook is to transform the feature-engineered dataset into a machine learning-ready format.

This includes:
- Defining the target variable
- Removing unnecessary features
- Splitting the dataset into training and testing sets
- Encoding categorical variables
- Scaling numerical features
- Handling class imbalance
- Exporting processed datasets and preprocessing artifacts for downstream models.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [3]:
df = pd.read_csv("..//data//processed//hr_feature_engineered.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 63 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1470 non-null   int64  
 1   Attrition                 1470 non-null   object 
 2   BusinessTravel            1470 non-null   object 
 3   DailyRate                 1470 non-null   int64  
 4   Department                1470 non-null   object 
 5   DistanceFromHome          1470 non-null   int64  
 6   Education                 1470 non-null   int64  
 7   EducationField            1470 non-null   object 
 8   EmployeeCount             1470 non-null   int64  
 9   EmployeeNumber            1470 non-null   int64  
 10  EnvironmentSatisfaction   1470 non-null   int64  
 11  Gender                    1470 non-null   object 
 12  HourlyRate                1470 non-null   int64  
 13  JobInvolvement            1470 non-null   int64  
 14  JobLevel

### Target Variable

Convert the target into binary labels.

In [5]:
df["Attrition"] = (
    df["Attrition"]
    .map({
        "No": 0,
        "Yes": 1
    })
)

df["Attrition"].value_counts()

Attrition
0    1233
1     237
Name: count, dtype: int64

### Remove Unnecessary Columns

These columns don't provide predictive value.

In [6]:
columns_to_drop = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df = df.drop(columns=columns_to_drop)

### Separate Features and Target

In [7]:
X = df.drop(columns="Attrition")

y = df["Attrition"]

### Train-Test Split

We'll use stratification because attrition is imbalanced (about 19.2% "Yes").

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### Verify Shapes

In [9]:
print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)

print("Training Target   :", y_train.shape)
print("Testing Target    :", y_test.shape)

Training Features : (1176, 58)
Testing Features  : (294, 58)
Training Target   : (1176,)
Testing Target    : (294,)


### Identify Feature Types

In [10]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

In [11]:
print(categorical_features)

print()

print(numerical_features)

['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'CareerStage', 'IncomeBand', 'CommuteBand', 'ExperienceBand', 'EngagementLevel', 'WellbeingLevel', 'CareerGrowthLevel', 'AttritionRiskLevel', 'LoyaltyLevel', 'ManagerStabilityLevel', 'ExperienceMaturity', 'TenureCommitment']

['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'PromotionDelay', 'OvertimeRisk', 'FrequentTraveler', 'LongCommute', 'ExperienceRatio', 'TenureRatio', 'EngagementScore', 'WellbeingScore', 'CareerGrowthScore', 'LoyaltyScore', 'ManagerStability', 'SalaryPercentile', 'IncomePerJo

### Create the Preprocessing Pipeline

Instead of manually encoding and scaling, we'll use a ColumnTransformer. This is the standard approach in scikit-learn and makes the preprocessing reproducible.

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

### Fit on Training Data Only

In [13]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

### Retrieve Feature Names

Since One-Hot Encoding creates new columns, let's recover their names.

In [14]:
encoded_feature_names = preprocessor.named_transformers_[
    "cat"
].get_feature_names_out(categorical_features)

all_feature_names = (
    numerical_features +
    list(encoded_feature_names)
)

### Convert Back to DataFrames

Working with DataFrames is much easier than NumPy arrays.

In [15]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=all_feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=all_feature_names,
    index=X_test.index
)

### Verify

In [16]:
print(X_train_processed.shape)

display(X_train_processed.head())

(1176, 114)


,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,PromotionDelay,OvertimeRisk,FrequentTraveler,LongCommute,ExperienceRatio,TenureRatio,EngagementScore,WellbeingScore,CareerGrowthScore,LoyaltyScore,ManagerStability,SalaryPercentile,IncomePerJobLevel,YoungProfessional,SeniorEmployee,AttritionRiskScore,BusinessTravel_Non-Travel,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Human Resources,Department_Research & Development,Department_Sales,EducationField_Human Resources,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,Gender_Female,Gender_Male,JobRole_Healthcare Representative,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single,OverTime_No,OverTime_Yes,CareerStage_Early Career,CareerStage_Experienced,CareerStage_Mid Career,CareerStage_Senior,IncomeBand_High,IncomeBand_Low,IncomeBand_Medium,IncomeBand_Very High,CommuteBand_Far,CommuteBand_Moderate,CommuteBand_Near,ExperienceBand_Expert,ExperienceBand_Intermediate,ExperienceBand_Junior,ExperienceBand_Senior,EngagementLevel_High,EngagementLevel_Low,EngagementLevel_Medium,EngagementLevel_Very High,WellbeingLevel_High,WellbeingLevel_Low,WellbeingLevel_Medium,WellbeingLevel_Very High,CareerGrowthLevel_Excellent,CareerGrowthLevel_High,CareerGrowthLevel_Low,CareerGrowthLevel_Medium,AttritionRiskLevel_Critical Risk,AttritionRiskLevel_High Risk,AttritionRiskLevel_Low Risk,AttritionRiskLevel_Moderate Risk,LoyaltyLevel_High,LoyaltyLevel_Low,LoyaltyLevel_Medium,LoyaltyLevel_Very High,ManagerStabilityLevel_High,ManagerStabilityLevel_Low,ManagerStabilityLevel_Medium,ManagerStabilityLevel_Very High,ExperienceMaturity_Beginner,ExperienceMaturity_Developing,ExperienceMaturity_Experienced,ExperienceMaturity_Veteran,TenureCommitment_High,TenureCommitment_Low,TenureCommitment_Moderate,TenureCommitment_Very High
1194,1.09,1.05,-0.90,1.06,-0.66,-0.91,1.80,1.76,-0.65,2.03,0.93,1.33,-0.34,-0.43,0.24,2.61,2.26,-0.61,0.34,-0.67,-0.63,-0.37,-0.62,-0.46,-0.64,-0.49,-0.58,2.17,-1.78,0.04,-0.36,1.23,-1.71,0.10,1.42,1.31,-0.60,1.87,-0.63,0.00,0.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00
128,-1.63,-0.52,-0.90,-1.86,0.26,1.69,0.37,-0.99,1.15,-0.86,0.68,-1.08,-0.34,-0.43,0.24,0.25,-1.07,-0.61,0.34,-0.83,-0.91,-0.06,-0.90,-0.46,-0.64,-0.49,-0.58,-0.98,-0.03,1.03,0.40,-0.90,-0.30,-0.52,-1.21,-0.59,1.68,-0.53,-1.39,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00
810,0.98,-0.99,-0.78,-1.86,-1.58,-0.66,0.37,1.76,0.25,2.35,0.17,0.12,-0.88,-0.43,1.16,0.25,1.49,0.19,0.34,0.81,1.34,0.57,1.35,-0.46,-0.64,-0.49,-0.58,1.38,-0.49,0.04,-1.11,0.33,-0.30,0.78,1.53,1.80,-0.60,1.87,-1.02,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.

### Check Class Distribution

Before handling imbalance, inspect the target distribution.

In [17]:
print("Training Target Distribution")

print(y_train.value_counts())

print()

print(
    (
        y_train.value_counts(normalize=True) * 100
    ).round(2)
)

Training Target Distribution
Attrition
0    986
1    190
Name: count, dtype: int64

Attrition
0   83.84
1   16.16
Name: proportion, dtype: float64


### Save the processed datasets

In [18]:
X_train_processed.to_csv(
    "..//data//processed//X_train_processed.csv",
    index=False
)

X_test_processed.to_csv(
    "..//data//processed//X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "..//data//processed//y_train.csv",
    index=False
)

y_test.to_csv(
    "..//data//processed//y_test.csv",
    index=False
)

### Save preprocessing artifacts

In [19]:
joblib.dump(
    preprocessor,
    "..//models//preprocessor.pkl"
)

joblib.dump(
    all_feature_names,
    "..//models//feature_names.pkl"
)

['..//models//feature_names.pkl']

### Final summary

In [20]:
print("=" * 70)
print("DATA PREPROCESSING COMPLETE")
print("=" * 70)

print(f"Original Dataset Shape      : {df.shape}")
print(f"Training Dataset Shape      : {X_train.shape}")
print(f"Testing Dataset Shape       : {X_test.shape}")
print(f"Processed Training Shape    : {X_train_processed.shape}")
print(f"Processed Testing Shape     : {X_test_processed.shape}")
print(f"Total Features After Encoding: {len(all_feature_names)}")

print("=" * 70)

DATA PREPROCESSING COMPLETE
Original Dataset Shape      : (1470, 59)
Training Dataset Shape      : (1176, 58)
Testing Dataset Shape       : (294, 58)
Processed Training Shape    : (1176, 114)
Processed Testing Shape     : (294, 114)
Total Features After Encoding: 114


## 16. Encode Categorical Variables

Machine learning algorithms require numerical input features. While the engineered dataset contains valuable categorical information for business analysis and Power BI dashboards, predictive models require these variables to be transformed into numeric representations.

One-Hot Encoding is applied to nominal categorical variables to preserve information without introducing artificial ordinal relationships. The resulting dataset will serve as the standardized input for all subsequent machine learning and deep learning models.

In [21]:
categorical_features = df.select_dtypes(
    include="object"
).columns.tolist()

categorical_features

['BusinessTravel',
 'Department',
 'EducationField',
 'Gender',
 'JobRole',
 'MaritalStatus',
 'OverTime',
 'CareerStage',
 'IncomeBand',
 'CommuteBand',
 'ExperienceBand',
 'EngagementLevel',
 'WellbeingLevel',
 'CareerGrowthLevel',
 'AttritionRiskLevel',
 'LoyaltyLevel',
 'ManagerStabilityLevel',
 'ExperienceMaturity',
 'TenureCommitment']

In [22]:
df = pd.get_dummies(
    df,
    columns=categorical_features,
    drop_first=False,
    dtype=int
)

In [23]:
df.head()

,Age,Attrition,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,PromotionDelay,OvertimeRisk,FrequentTraveler,LongCommute,ExperienceRatio,TenureRatio,EngagementScore,WellbeingScore,CareerGrowthScore,LoyaltyScore,ManagerStability,SalaryPercentile,IncomePerJobLevel,YoungProfessional,SeniorEmployee,AttritionRiskScore,BusinessTravel_Non-Travel,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Human Resources,Department_Research & Development,Department_Sales,EducationField_Human Resources,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,Gender_Female,Gender_Male,JobRole_Healthcare Representative,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single,OverTime_No,OverTime_Yes,CareerStage_Early Career,CareerStage_Experienced,CareerStage_Mid Career,CareerStage_Senior,IncomeBand_High,IncomeBand_Low,IncomeBand_Medium,IncomeBand_Very High,CommuteBand_Far,CommuteBand_Moderate,CommuteBand_Near,ExperienceBand_Expert,ExperienceBand_Intermediate,ExperienceBand_Junior,ExperienceBand_Senior,EngagementLevel_High,EngagementLevel_Low,EngagementLevel_Medium,EngagementLevel_Very High,WellbeingLevel_High,WellbeingLevel_Low,WellbeingLevel_Medium,WellbeingLevel_Very High,CareerGrowthLevel_Excellent,CareerGrowthLevel_High,CareerGrowthLevel_Low,CareerGrowthLevel_Medium,AttritionRiskLevel_Critical Risk,AttritionRiskLevel_High Risk,AttritionRiskLevel_Low Risk,AttritionRiskLevel_Moderate Risk,LoyaltyLevel_High,LoyaltyLevel_Low,LoyaltyLevel_Medium,LoyaltyLevel_Very High,ManagerStabilityLevel_High,ManagerStabilityLevel_Low,ManagerStabilityLevel_Medium,ManagerStabilityLevel_Very High,ExperienceMaturity_Beginner,ExperienceMaturity_Developing,ExperienceMaturity_Experienced,ExperienceMaturity_Veteran,TenureCommitment_High,TenureCommitment_Low,TenureCommitment_Moderate,TenureCommitment_Very High
0,41,1,1102,1,2,2,94,3,2,4,5993,19479,8,11,3,1,0,8,0,1,6,4,0,5,0,1,0,0,0.20,0.75,2.00,0.33,0.42,0.67,0.71,0.62,2996.50,0,0,0.40,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
1,49,0,279,8,1,3,61,2,2,2,5130,24907,1,23,4,4,1,10,3,3,10,7,1,7,0,0,1,0,0.20,1.00,2.33,1.33,0.68,0.91,0.64,0.52,2565.00,0,1,0.20,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,1
2,37,1,1373,2,2,4,92,2,1,3,2090,2396,6,15,3,2,0,7,3,3,0,0,0,0,0,1,0,0,0.19,0.00,2.33,1.67,0.43,0.00,0.00,0.05,2090.00,0,0,0.33,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0
3,33,0,1392,3,4,4,56,3,1,3,2909,23159,1,11,3,3,0,8,3,3,8,7,3,0,0,1,1,0,0.24,1.00,3.00,1.67,0.27,0.89,0.00,0.25,2909.00,0,0,0.33,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,1
4,27,0,591,2,1,1,40,3,1,2,3468,16632,9,12,3,4,1,6,3,3,2,2,2,2,0,0,0,0,0.22,0.33,2.00,0.67,0.31,0.29,0.67,0.32,3468.00,1,0,0.20,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0


In [24]:
remaining = df.select_dtypes(include="object").columns.tolist()

print(remaining)

[]


### Verify Dataset

In [25]:
print(df.shape)

(1470, 115)


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Columns: 115 entries, Age to TenureCommitment_Very High
dtypes: float64(10), int64(105)
memory usage: 1.3 MB


In [27]:
df.isna().sum().sum()

np.int64(0)

In [29]:
from pathlib import Path

In [30]:
processed_path = Path("..//data//processed")

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

In [31]:
processed_file = processed_path / "hr_feature_engineered.csv"

df.to_csv(
    processed_file,
    index=False
)

print(f"Dataset saved to:\n{processed_file}")

Dataset saved to:
..\data\processed\hr_feature_engineered.csv


### Final Verification

In [32]:
verification = pd.read_csv(processed_file)

verification.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Columns: 115 entries, Age to TenureCommitment_Very High
dtypes: float64(10), int64(105)
memory usage: 1.3 MB


In [33]:
feature_names = pd.DataFrame({
    "Feature": df.columns
})

feature_names.to_csv(
    "..//reports//tables//feature_names.csv",
    index=False
)

print("Feature names saved successfully.")

Feature names saved successfully.


# Conclusion

This notebook transformed the cleaned HR dataset into a machine-learning-ready dataset through feature engineering and categorical encoding.

The completed feature engineering pipeline included:

- Creation of business-oriented composite scores
- Employee engagement and wellbeing metrics
- Career growth and loyalty indicators
- Attrition risk scoring
- Executive-friendly categorical bands
- One-Hot Encoding of categorical variables

The resulting processed dataset contains only numerical features, making it suitable for machine learning, deep learning, explainable AI, and deployment pipelines while preserving engineered business insights for executive analytics.

This processed dataset will be used consistently across all subsequent modeling notebooks, ensuring a standardized and reproducible workflow.